# Task 02 – Data Cleaning and Preparation
## Superstore Sales Dataset | SkillCraft Technology

**Objective:** Inspect, clean, validate and export the supplied sales dataset using Python and Pandas.

**Important source-format finding:** The source date strings use `DD/MM/YYYY`. This format is specified explicitly during conversion to prevent ambiguous or incorrect date interpretation.

**Cleaning rules:** Remove exact duplicate rows only; trim surrounding whitespace from text fields; convert dates and Sales to appropriate data types; retain unknown Postal Code values as missing; create date features for analysis.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Put the extracted source CSV in the same folder as this notebook.
file_path = Path("train.csv")
sales = pd.read_csv(file_path)
print("Dataset shape:", sales.shape)
sales.head()


## 1. Inspect structure and data types

In [ ]:
sales.info()
sales.describe(include="all").T


## 2. Identify missing values and duplicate rows

In [ ]:
print("Missing values by column:")
print(sales.isna().sum()[sales.isna().sum() > 0])
print("\nExact duplicate rows:", sales.duplicated().sum())


## 3. Clean the dataset

In [ ]:
cleaned = sales.drop_duplicates().copy()

# Standardize text by removing leading/trailing spaces
for col in cleaned.select_dtypes(include=["object"]).columns:
    cleaned[col] = cleaned[col].astype("string").str.strip()

# Source dates are DD/MM/YYYY
for col in ["Order Date", "Ship Date"]:
    cleaned[col] = pd.to_datetime(
        cleaned[col], format="%d/%m/%Y", errors="coerce"
    )

cleaned["Sales"] = pd.to_numeric(cleaned["Sales"], errors="coerce")

# Postal Code is an identifier, stored as nullable integer
cleaned["Postal Code"] = pd.to_numeric(
    cleaned["Postal Code"], errors="coerce"
).astype("Int64")

print("Cleaned shape:", cleaned.shape)


## 4. Validate cleaning results

In [ ]:
print("Remaining exact duplicates:", cleaned.duplicated().sum())
print("Missing values after cleaning:")
print(cleaned.isna().sum()[cleaned.isna().sum() > 0])
print("\nUpdated data types:")
print(cleaned.dtypes)


## 5. Create date features

In [ ]:
cleaned["Order Year"] = cleaned["Order Date"].dt.year.astype("Int64")
cleaned["Order Month"] = cleaned["Order Date"].dt.month.astype("Int64")
cleaned["Order Month Name"] = cleaned["Order Date"].dt.strftime("%B").astype("string")
cleaned["Year-Month"] = cleaned["Order Date"].dt.to_period("M").astype("string")

cleaned[["Order Date", "Order Year", "Order Month",
         "Order Month Name", "Year-Month"]].head()


## 6. Export cleaned data

In [ ]:
output_file = Path("Task_02_Cleaned_Superstore_Sales.csv")
cleaned.to_csv(output_file, index=False)
print("Cleaned CSV saved to:", output_file.resolve())


## 7. Conclusion

The dataset was profiled and prepared for further analysis. Exact duplicate records were checked, text fields were trimmed, date and sales columns were converted to suitable data types, and date-derived columns were added. The source date format was correctly interpreted as DD/MM/YYYY. Missing postal codes were retained as null because the correct values cannot be inferred reliably.

**Data integrity note:** No missing postal codes or sales values were replaced with invented values.
